# 03 - Train aLoRA on macOS MPS

Run this notebook only after the ordinary LoRA baseline passes its quality gate. It preserves the same data and task while adding an activation marker.

The exact invocation token sequence must remain identical during training, direct PEFT evaluation, Granite Switch composition, and composed inference.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import torch


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Repository root not found")


ROOT = find_repo_root(Path.cwd().resolve())
ADAPTER_NAME = "my_adapter"
BASE_MODEL = "ibm-granite/granite-4.1-3b"
INVOCATION = f"<{ADAPTER_NAME}>"
DATA_DIR = ROOT / "workspaces" / ADAPTER_NAME
OUTPUT_DIR = ROOT / "outputs" / "my-library" / ADAPTER_NAME / "granite-4.1-3b" / "alora"
RUN_TRAINING = False

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for this Mac training notebook")
print("Invocation:", INVOCATION)
print("Output:", OUTPUT_DIR)

## Validate token-level invocation behavior

This cell downloads only tokenizer assets. Enable it by changing `CHECK_TOKENIZER` after confirming the base-model ID.

In [ ]:
CHECK_TOKENIZER = False
if CHECK_TOKENIZER:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    invocation_ids = tokenizer.encode(INVOCATION, add_special_tokens=False)
    assert invocation_ids
    assert tokenizer.encode(INVOCATION[0], add_special_tokens=False) == invocation_ids[:1]
    assert tokenizer.encode(INVOCATION[1:], add_special_tokens=False) == invocation_ids[1:]
    print("Invocation token IDs:", invocation_ids)
else:
    print("Tokenizer download disabled")

## Build the aLoRA command

The marker is appended to each user message immediately before the assistant generation boundary. Granite Switch creates its internal control token later during composition; do not add that internal token to the dataset.

In [ ]:
command = [
    sys.executable,
    str(ROOT / "scripts" / "train_adapter.py"),
    "--technology",
    "alora",
    "--invocation",
    INVOCATION,
    "--base-model",
    BASE_MODEL,
    "--train-file",
    str(DATA_DIR / "train.jsonl"),
    "--validation-file",
    str(DATA_DIR / "validation.jsonl"),
    "--output-dir",
    str(OUTPUT_DIR),
    "--rank",
    "8",
    "--alpha",
    "16",
    "--target-modules",
    "q_proj,v_proj",
    "--epochs",
    "1",
    "--batch-size",
    "1",
    "--gradient-accumulation-steps",
    "8",
    "--max-length",
    "256",
    "--gradient-checkpointing",
]
print(" ".join(map(str, command)))

In [ ]:
required = [DATA_DIR / "train.jsonl", DATA_DIR / "validation.jsonl", DATA_DIR / "io.yaml"]
missing = [path for path in required if not path.is_file()]

if RUN_TRAINING:
    if not CHECK_TOKENIZER:
        raise RuntimeError("Run the tokenizer invariant check before aLoRA training")
    if missing:
        raise FileNotFoundError(f"Missing required inputs: {missing}")
    subprocess.run(command, cwd=ROOT, check=True)
    torch.mps.synchronize()
    torch.mps.empty_cache()
else:
    print("Training disabled. Complete the LoRA quality gate, then enable this cell.")

## Validate the aLoRA artifact

The saved `adapter_config.json` must contain a non-empty integer list named `alora_invocation_tokens`.

In [ ]:
if RUN_TRAINING:
    io_text = (DATA_DIR / "io.yaml").read_text(encoding="utf-8")
    (OUTPUT_DIR / "io.yaml").write_text(io_text, encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "granite_adapter_guide", "validate-adapter", str(OUTPUT_DIR)],
        cwd=ROOT,
        check=True,
    )